# JS05: Lab 3
## Klasifikasi Naive Bayes dengan data SMS

## Persiapan data
Dataset SMS disimpan di folder notebook. Jika belum ada, notebook mengunduhnya otomatis.

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen

data_path = Path("spam.csv")
if not data_path.exists():
    request = Request("https://1812311909-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2FYHOv2XH8FJMJSMsnhwNa%2Fuploads%2F5SOal6OzJik1AsEMRu5M%2Fspam.csv?alt=media&token=29d9bc0d-7698-4e86-9404-aa3559f1ed31", headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request) as response:
        data_path.write_bytes(response.read())
print(f"Dataset siap: {data_path.resolve()}")

Dataset siap: D:\Machine Learning\JS05\spam.csv


## Langkah 1: Load data

In [2]:
import pandas as pd

df = pd.read_csv(data_path, encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## Langkah 2: Preprocessing

In [3]:
df = df.drop(df.iloc[:, 2:], axis=1)
df = df.rename(columns={"v1": "Labels", "v2": "SMS"})
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Inspeksi data

In [4]:
print(df["Labels"].value_counts())
print()
df.info()
print()
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  5572 non-null   str  
 1   SMS     5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


## Encoding label

In [5]:
new_labels = {"spam": 1, "ham": 0}
df["Labels"] = df["Labels"].map(new_labels)
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


## Memisahkan fitur dan label

In [6]:
X = df["SMS"].values
y = df["Labels"].values

## Langkah 3: Ekstraksi fitur dengan CountVectorizer

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50
)
bow = CountVectorizer()
X_train = bow.fit_transform(X_train)
X_test = bow.transform(X_test)
print(len(bow.get_feature_names_out()))
print(f"Dimensi data: {X_train.shape}")

7727
Dimensi data: (4457, 7727)


## Langkah 4: Training dan evaluasi model

In [8]:
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred_train = mnb.predict(X_train)
y_pred_test = mnb.predict(X_test)
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)
print(f"Hasil akurasi data train: {acc_train}")
print(f"Hasil akurasi data test: {acc_test}")

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229
